# Carga de datos

Se importan las librerías requeridas para la carga de datos:
- yfinance: para la extracción de los datos mediante la API de yahoo finance
- pandas: para la mnipulación de datos
- warnings: para manejar mensajes de advertencia

In [1]:
import yfinance as yf
import pandas as pd
import warnings
warnings.filterwarnings("ignore")


Se configuran las fechas de inicio y fin para la extracción de datos en ese periodo

In [ ]:
# se configuran las fechas de inicio y fin
start_date = '2021-01-01'
end_date = '2025-12-31'
#end_date = '2025-05-30'
#start_date = "2015-01-01" #test se pueden borrar
#end_date = "2023-01-01" #test se pueden borrar

## Datos series temporales

Siguiendo la metodología del artículo "Nvidia's stock returns prediction using machine learning techniques for time series forecasting problem", se toman los datos de las siguientes series (incluyendo competidores direct y partners):
- precios de NVidia (close, high low, open, volume) ✔️
- precios de AMD (close, high low, open, volume) ✔️
- BITCOIN USD (close, high low, open, volume) ✔️
- Ubisoft (close, high low, open, volume) ✔️
- ATVI (close, high low, open, volume) - falta
- S&P500 (close, high low, open, volume) ✔️
- NASDAQ100(close, high low, open, volume) ✔️


In [5]:
# recorre la lista de series y descarga un csv por cada una
tickets = [
           "AAPL", #Apple
           "NVDA", # Nvidia 
           "^SPX", #S&P500
           "AMD", # AMD
           "BTC-USD", # Bitcoin
           "^NDX", # NAsdaq100
           "^VIX", # indice de volatilidad
           "UBSFY", # Ubisoft
           "EC" #ecopetrol
           ] 
for ticket in tickets: # se recorre cada ticket
    ticket_cleaned = ticket[1:] if ticket.startswith('^') else ticket # se crea el ticket con el nombre limpio (para definir las columnas)
    try:
        df_data_close = yf.download(ticket, start=start_date, end=end_date) # se descargan los datos de la divisa seleccionada en el periodo dado (con los campos mencionados antes)
        df_data_close.columns = df_data_close.columns.to_flat_index() # se aplana el df para que no tenga mas de un encabezado
        df_data_close = df_data_close.rename( # renombrado de columnas
            columns={
                ('Close', ticket): f'{ticket_cleaned}_Close',
                ('High', ticket):  f'{ticket_cleaned}_High',
                ('Low', ticket):  f'{ticket_cleaned}_Low',
                ('Open', ticket):  f'{ticket_cleaned}_Open',
                ('Volume', ticket):  f'{ticket_cleaned}_Volume',
        }
        )
        df_data_close = df_data_close.reset_index() # se resetea la columna de indice
        df_data_close = df_data_close.sort_values(by="Date", ascending=True) # se ordena el df por fecha
        
        print("datos ok")
    except Exception as e: 
        raise SystemExit(f"Error cargando los datos: {e}")
    
    df_data_close.to_csv(f"csv/{ticket_cleaned}_data.csv", index=False)  # Guardar como CSV
    print(f"Archivo {ticket_cleaned}_data.csv guardado correctamente.")



[*********************100%***********************]  1 of 1 completed


datos ok
Archivo AAPL_data.csv guardado correctamente.


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


datos ok
Archivo NVDA_data.csv guardado correctamente.
datos ok
Archivo SPX_data.csv guardado correctamente.
datos ok
Archivo AMD_data.csv guardado correctamente.


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


datos ok
Archivo BTC-USD_data.csv guardado correctamente.
datos ok
Archivo NDX_data.csv guardado correctamente.
datos ok
Archivo VIX_data.csv guardado correctamente.


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


datos ok
Archivo UBSFY_data.csv guardado correctamente.
datos ok
Archivo EC_data.csv guardado correctamente.


Esta franja es para, a partir de los df creados anteriores, crear dataframes de spark y almacenarlos en un lakehouse (implementación en Microoft Fabric)

In [6]:
    # se crea un df de spark y se guarda la tabla en el lakehouse
    # df_data_close_spark = spark.createDataFrame(df_data_close)
    # df_data_cloee_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "True").save(f"Tables/{ticket_cleaned}_data")